# Cross-branch transfer matrix — direct-DPO branches

Adds the **training-history axis** (safety-SFT-mediated vs direct-to-DPO) to the
existing corpus axis (Alpaca vs Dolly). Four branches now:

| branch | pre-DPO | post-DPO | corpus | history |
|---|---|---|---|---|
| A | M2 | M3 | Alpaca | m2-mediated |
| B | M2_alt | M3_alt | Dolly | m2-mediated |
| A_direct | **M1** | M3_direct | Alpaca | **direct** |
| B_direct | **M1_alt** | M3_direct_alt | Dolly | **direct** |

**A→B and B→A are done.** This notebook runs the rest, config-driven.

## How to use

Set `PAIRS` in the config cell to one session's worth of `(source, target)`
directions. Each direction: assemble its deltas → (gate if the target has no
full gate yet) → Stage-2 core → **bank**. Fully resumable — the runner skips a
unit whose output exists, and section 0.5 restores prior banks.

**Session 1 (tonight, ~5h): the history axis, both corpora, reciprocal** —
```
PAIRS = [("A","A_direct"), ("A_direct","A"), ("B","B_direct"), ("B_direct","B")]
```
**Session 2 (~5h): both-axes pairs + within-direct corpus axis + judges** —
```
PAIRS = [("A","B_direct"), ("B_direct","A"), ("B","A_direct"), ("A_direct","B"),
         ("A_direct","B_direct"), ("B_direct","A_direct")]
```

Targets M2 (A) and M2_alt (B) already have a full 3-coefficient gate from the
A↔B run, so directions into them only regenerate the B2/B3 anchors that
`analyze_stage2` needs. Targets M1 / M1_alt get a coef-1.0 within-branch check
(the full stop-gate was a project-level decision already satisfied by A/B).

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
REPO_URL='https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR='/content/dpo-safety-representations'
BRANCH='agent/c-quadrant-end-to-end-e0e2317a'
PINNED='c784139b156910b717a73ae7ae5e1f2cca803c98'
if not os.path.exists(REPO_DIR):
    subprocess.run(["git","clone","-b",BRANCH,REPO_URL,REPO_DIR],check=True)
os.chdir(REPO_DIR)
subprocess.run(["git","fetch","origin"],check=True); subprocess.run(["git","checkout",PINNED],check=True)
print("HEAD",subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip())

### 0.2 HF auth — needed for the judges section only

In [ ]:
import os
try:
    from google.colab import userdata
    from huggingface_hub import login
    _t=userdata.get('HF_TOKEN'); os.environ['HF_TOKEN']=_t; login(token=_t)
    print('HF login OK')
except Exception as e:
    print('HF NOT authenticated:',repr(e),'-- generation sections still run')

### 0.3 Apply the patch — upload the LATEST `crossbranch_p0_patch.zip` (the build with the 4-branch `BRANCHES`)

In [ ]:
from google.colab import files
import zipfile, io
up=files.upload(); assert len(up)==1
name,data=next(iter(up.items()))
with zipfile.ZipFile(io.BytesIO(data)) as z:
    ns=z.namelist()
    assert all(n.startswith(('src/crossbranch/','tests/crossbranch/')) for n in ns)
    z.extractall('.')
print("applied",len(ns),"files")

In [ ]:
!pip -q install -r requirements.txt
!pip -q install bitsandbytes
!pip uninstall -y torchao || true
!nvidia-smi

### 0.5 Restore prior results — upload every `crossbranch_*results*.zip` / `crossbranch_finish_*.zip` / `crossbranch_matrix_*.zip` you have

In [ ]:
import zipfile, io, os, glob
from google.colab import files
os.makedirs("results/crossbranch", exist_ok=True)

# Upload every result zip you have downloaded so far -- works in ANY Google
# account, no folder sharing needed:
#   - crossbranch_prior_results.zip / crossbranch_finish_*.zip   (the A<->B run)
#   - crossbranch_matrix_*.zip  (newest is cumulative; that one alone covers the
#     matrix directions -- extra zips are harmless, identical content)
print("upload your result zips (multi-select ok); cancel only on a true first run:")
for name, data in files.upload().items():
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        z.extractall("results/crossbranch")
    print("restored", name)

raw = [p for p in sorted(glob.glob("results/crossbranch/raw/crossbranch_*.json"))
       if not p.endswith("_binding.json")]
print("\n%d raw response files present" % len(raw))

### 0.6 Copy activations + directions for ALL 8 stages from Drive

In [ ]:
RESULTS_SOURCE_DIR='/content/drive/MyDrive/dpo_v2/results'
import shutil
from pathlib import Path
src=Path(RESULTS_SOURCE_DIR); assert (src/'activations').exists(), "fix RESULTS_SOURCE_DIR"
STAGES=('M2','M3','M2_alt','M3_alt','M1','M1_alt','M3_direct','M3_direct_alt')
copied,missing=[],[]
d=Path('results/activations'); d.mkdir(parents=True,exist_ok=True)
for s in STAGES:
    for suf in ('_final.npy','_pooled.npy','_metadata.json','_metadata_binding.json'):
        f=src/'activations'/f'{s}{suf}'; (copied if f.exists() else missing).append(f.name)
        if f.exists(): shutil.copy2(f,d/f'{s}{suf}')
dd=Path('results/refusal_direction'); dd.mkdir(parents=True,exist_ok=True)
for s in ('M3','M3_alt','M1','M1_alt','M3_direct','M3_direct_alt'):
    for suf in ('_direction_654.npy','_direction_654_binding.json'):
        f=src/'refusal_direction'/f'{s}{suf}'; (copied if f.exists() else missing).append(f.name)
        if f.exists(): shutil.copy2(f,dd/f'{s}{suf}')
print("copied",len(copied)); [print("  MISSING",m) for m in missing]
assert not missing, "some stages missing on Drive -- check RESULTS_SOURCE_DIR"

### 0.7 Preflight — all 8 stages bind to the frozen benchmark

In [ ]:
import json
from pathlib import Path
import numpy as np
from src.v2_io import load_run_inputs, identity_snapshot, load_json
bp,bsha,sp,ssha=load_run_inputs(None,None,'logs/direction_split_manifest.json')
rows=[json.loads(l) for l in Path(bp).read_text(encoding='utf-8').splitlines() if l.strip()]
snap=identity_snapshot(rows); ok=True
for s in ('M2','M3','M2_alt','M3_alt','M1','M1_alt','M3_direct','M3_direct_alt'):
    a=np.load(f'results/activations/{s}_final.npy',mmap_mode='r')
    good=(a.shape[0]==len(rows) and load_json(f'results/activations/{s}_metadata.json')==snap
          and load_json(f'results/activations/{s}_metadata_binding.json').get('benchmark_sha256')==bsha)
    ok&=good; print(f"  {s:14s} {str(a.shape):18s} {'PASS' if good else 'FAIL'}")
for s in ('M3','M3_alt','M1','M1_alt','M3_direct','M3_direct_alt'):
    n=float(np.linalg.norm(np.load(f'results/refusal_direction/{s}_direction_654.npy')[24]))
    ok&=abs(n-1)<1e-3; print(f"  dir {s:12s} L24 norm={n:.6f}")
assert ok, "preflight FAILED"; print("\nAll PASS.")

In [ ]:
!python -m pytest tests/crossbranch -q

---
# CONFIG — set PAIRS for this session, then run the loop below

In [ ]:
# Session 1 (history axis, both corpora, reciprocal):
PAIRS = [("A","A_direct"), ("A_direct","A"), ("B","B_direct"), ("B_direct","B")]

# Session 2 (both-axes + within-direct corpus axis):
# PAIRS = [("A","B_direct"), ("B_direct","A"), ("B","A_direct"), ("A_direct","B"),
#          ("A_direct","B_direct"), ("B_direct","A_direct")]

STAGE2_CORE = ("xfer_delta_source_identity xfer_delta_source_shuf_wq "
               "xfer_delta_source_normmatched xfer_delta_source_dosematched "
               "dir_source_matched dir_target_matched")
HAVE_FULL_GATE = {"A", "B"}   # M2 / M2_alt already have a full 3-coef gate from A<->B

import shutil, os
from google.colab import files


def bank(name):
    """Zip ONLY raw/ + analysis/ + judges/ + manifests/ (compact, cumulative)
    and download it. deltas_*/ and raw/shards/ are excluded -- they regenerate
    from CPU in seconds and are not needed for resume or local analysis. The
    zip is CUMULATIVE: keep only the newest one you download."""
    stage = "/content/_bank"
    if os.path.isdir(stage):
        shutil.rmtree(stage)
    for sub in ("raw", "analysis", "judges", "manifests"):
        s = "results/crossbranch/" + sub
        if os.path.isdir(s):
            shutil.copytree(s, stage + "/" + sub,
                            ignore=shutil.ignore_patterns("shards"))
    z = shutil.make_archive("/content/" + name, "zip", stage)
    print("  " + name + ".zip  %.1f MB" % (os.path.getsize(z) / 1e6))
    try:
        files.download(z)
    except Exception as e:
        print("  (download failed:", e, "-- re-run bank('" + name + "'))")


print("this session:", PAIRS)

## The run loop

For each `(src, tgt)`:
1. assemble deltas + decomposition into `deltas_<src>to<tgt>/`
2. if `tgt` has no full gate: generate `baseline_target reference_target
   own_delta_target own_normmatched_random` at coef 1.0 (a within-branch check)
   — else just `baseline_target reference_target` (the B2/B3 anchors
   `analyze_stage2` needs)
3. Stage-2 core at coef 1.0
4. **bank** (download `crossbranch_matrix_<src>to<tgt>.zip`)

Re-running is safe: finished units are skipped. If a direction dies mid-way,
re-run this cell — it resumes from the banked + on-disk state.

In [ ]:
import subprocess, sys

for src, tgt in PAIRS:
    tag = src + "to" + tgt
    ddir = "results/crossbranch/deltas_" + tag
    print("\n" + "=" * 60 + "\n" + tag + "\n" + "=" * 60, flush=True)

    subprocess.run([sys.executable, "-m", "src.crossbranch.delta",
                    "--stage2", "--decomposition",
                    "--source-branch", src, "--target-branch", tgt,
                    "--out-dir", ddir], check=True)

    gate_conds = (["baseline_target", "reference_target"] if tgt in HAVE_FULL_GATE
                  else ["baseline_target", "reference_target",
                        "own_delta_target", "own_normmatched_random"])
    all_conds = gate_conds + STAGE2_CORE.split()

    for n, cond in enumerate(all_conds, 1):
        print("  [%s] %d/%d  %s" % (tag, n, len(all_conds), cond), flush=True)
        subprocess.run([sys.executable, "-m", "src.crossbranch.runner",
                        "--allow-stage2", "--source-branch", src,
                        "--target-branch", tgt, "--deltas-dir", ddir,
                        "--conditions", cond, "--coefficients", "1.0"], check=True)

    bank("crossbranch_matrix_" + tag)
    print("  " + tag + " DONE + banked", flush=True)

print("\nALL DIRECTIONS DONE")

---
# JUDGES — quadrant C only, every arm present  (run last, ~1h)

Scores every `xfer_*` / `own_delta_target` / model-condition arm on quadrant C,
for every direction present in `results/crossbranch/raw/`. Split per direction
tag so a mid-run death never loses a completed one.

In [ ]:
import subprocess, sys, glob, json, pathlib
from src.crossbranch.build_judge_manifest import parse_raw_filename

tags = sorted({parse_raw_filename(pathlib.Path(p).stem)[0]
               for p in glob.glob("results/crossbranch/raw/crossbranch_*.json")
               if not p.endswith("_binding.json")})
print("directions to judge:", tags)

for tag in tags:
    prev = sorted(glob.glob("results/crossbranch/judges/behavioral_judges_*%s*.json" % tag))
    if prev:
        st = json.loads(pathlib.Path(prev[-1]).read_text(encoding="utf-8")).get("judge_status", {})
        if st.get("strong_reject") == "scored" and st.get("wildguard") == "scored":
            print(tag, ": already scored")
            continue
    subprocess.run([sys.executable, "-m", "src.crossbranch.build_judge_manifest",
                    "--quadrants", "C", "--directions", tag], check=True)
    cmd = [sys.executable, "-m", "src.analysis.behavioral_judges",
           "--response-manifest", "results/crossbranch/manifests/crossbranch_judge_manifest.json",
           "--out-dir", "results/crossbranch/judges", "--run-live", "--scope", "all"]
    if prev:
        cmd += ["--resume-from", prev[-1]]
    subprocess.run(cmd, check=True)
    fresh = [x for x in sorted(glob.glob("results/crossbranch/judges/behavioral_judges_*.json"))
             if not any(t in x for t in tags)]
    if fresh:
        newn = fresh[-1].replace(".json", "_" + tag + ".json")
        pathlib.Path(fresh[-1]).rename(newn)
        print("tagged", newn)
    bank("crossbranch_matrix_judges_" + tag)

print("\nJUDGES DONE")

---
## STOP — local CPU

Unzip the latest `crossbranch_matrix_*.zip` into `results/crossbranch/`, then
for each new direction `<tag>`:

```
python -m src.crossbranch.analyze --source-branch <src> --target-branch <tgt>       # gate (if run)
python -m src.crossbranch.analyze_stage2 --source-branch <src> --target-branch <tgt>
```

then the matrix readout (a helper will be added):

```
python -m src.crossbranch.compare_directions --first AtoB --second BtoA
# ... pairwise per axis; a matrix summariser folds them into one table
python -m src.crossbranch.analyze_judges --judge-file results/crossbranch/judges/behavioral_judges_<ts>_<tag>.json
```

Expected per direction: `crossbranch_<tag>_stage2_analysis.json` with the 6-8
arms + contrasts; for M1 / M1_alt targets also `crossbranch_<tag>_analysis.json`
(the coef-1.0 within-branch check).